#  Egyptian Movies Dataset - Preprocessing


## Preprocessing Steps:
1. **Load Data**: Import CSV file
2. **Lowercase Conversion**: Standardize text
3. **Tokenization**: Split text into words
4. **Stopword Removal**: Remove common words
5. **Stemming**: Apply Snowball stemmers
6. **Save Results**: Export preprocessed data


## Import Required Libraries

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

## Download NLTK Resources

In [ ]:
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

True

## Configuration

In [ ]:
INPUT_FILE="/content/egyptian_movies.csv"
OUTPUT_FILE="../output/egyptian_movies_preprocessed.csv"
TEXT_COLUMNS=['title', 'abstract']

## Step 1: Load Data

In [ ]:
df=pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} movies")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Loaded 5036 movies

Columns: ['doc_id', 'title', 'type', 'year', 'rating', 'votes', 'popularity', 'abstract']

First few rows:


,doc_id,title,type,year,rating,votes,popularity,abstract
0,1,The Venal Gentlemen,movie,1983.0,6.000,1,8.4039,"Mahmoud, a simple employee of the Ministry of ..."
1,2,The Passage,movie,2019.0,7.042,24,5.0687,"A platoon of Commandos’ soldiers, lead by a fe..."
2,3,Karmouz War,movie,2018.0,6.000,50,4.4222,"Alexandria, Egypt, 1940. Three young Egyptians..."
3,4,The Emigrant,movie,1994.0,5.684,19,4.1952,The biblical tale of Joseph is told from an Eg...
4,5,The Swarm,movie,2024.0,4.200,5,3.7279,Based on actual events and the efforts of the ...


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDataset Info:")
df.info()

Missing values per column:
doc_id          0
title           0
type            0
year          202
rating          0
votes           0
popularity      0
abstract        0
dtype: int64

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5036 entries, 0 to 5035
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   doc_id      5036 non-null   int64  
 1   title       5036 non-null   object 
 2   type        5036 non-null   object 
 3   year        4834 non-null   float64
 4   rating      5036 non-null   float64
 5   votes       5036 non-null   int64  
 6   popularity  5036 non-null   float64
 7   abstract    5036 non-null   object 
dtypes: float64(3), int64(2), object(3)
memory usage: 314.9+ KB


## Step 2: Combine Text Fields


In [ ]:
def combine_text_fields(row):
    """Combine multiple text fields into one"""
    text_parts=[]
    if pd.notna(row['title']):
        text_parts.append(str(row['title']))
    if pd.notna(row['abstract']):
        text_parts.append(str(row['abstract']))
    return ' '.join(text_parts)
df['combined_text']=df.apply(combine_text_fields, axis=1)

#small example
print("\nSample combined text:")
print(df['combined_text'].iloc[0])


Sample combined text:
The Venal Gentlemen Mahmoud, a simple employee of the Ministry of Health, loves lawyer "Amal", who trains in the office of her teacher, "Ali Hammad", Mahmoud "travels" to Kuwait to improve his financial situation. Her wealthy professor escapes chasing her cousin, Fathi officer, Mahmoud settles in Cairo after returning from abroad and renews his hope to marry Amal after her husband's death. Bribery from "Ibrahim Al-Fakharani" to facilitate the process of importing rotten food for "Al-Fakharani". Amal defends him and accuses her cousin of fabricating the accusation. Mahmoud's order is revealed, he collapses and confesses that he is already bribed after he reaches his home to find that his brother, sister and children died after eating the canned food that he admitted to the country.


## Step 3: Lowercase Conversion

In [ ]:
lowercase_text=[]
for text in df['combined_text']:
    if pd.isna(text):
        lowercase_text.append("")
    else:
        lowercase_text.append(str(text).lower())
df['lowercase_text']=lowercase_text

#small example
print("Before:")
print(df['combined_text'].iloc[0])
print("After:")
print(df['lowercase_text'].iloc[0])

Before:
The Venal Gentlemen Mahmoud, a simple employee of the Ministry of Health, loves lawyer "Amal", who trains in the office of her teacher, "Ali Hammad", Mahmoud "travels" to Kuwait to improve his financial situation. Her wealthy professor escapes chasing her cousin, Fathi officer, Mahmoud settles in Cairo after returning from abroad and renews his hope to marry Amal after her husband's death. Bribery from "Ibrahim Al-Fakharani" to facilitate the process of importing rotten food for "Al-Fakharani". Amal defends him and accuses her cousin of fabricating the accusation. Mahmoud's order is revealed, he collapses and confesses that he is already bribed after he reaches his home to find that his brother, sister and children died after eating the canned food that he admitted to the country.
After:
the venal gentlemen mahmoud, a simple employee of the ministry of health, loves lawyer "amal", who trains in the office of her teacher, "ali hammad", mahmoud "travels" to kuwait to improve his 

## Step 4: Tokenization (Split into words)

In [ ]:
tokens_list=[]
for text in df['lowercase_text']:
    # Use regex to extract words (alphanumeric)
    words=re.findall(r'\w+', text)
    tokens_list.append(words)
df['tokens']=tokens_list

#small example
print(f"\nSample tokens (first 20):")
print(df['tokens'].iloc[0][:20])


Sample tokens (first 20):
['the', 'venal', 'gentlemen', 'mahmoud', 'a', 'simple', 'employee', 'of', 'the', 'ministry', 'of', 'health', 'loves', 'lawyer', 'amal', 'who', 'trains', 'in', 'the', 'office']


## Step 5: Stopword Removal

In [ ]:
stop_words=set(stopwords.words('english'))
print(f"Number of stopwords: {len(stop_words)}")

no_stopwords_list=[]
for tokens in df['tokens']:
    filtered_words=[word for word in tokens if word.lower() not in stop_words]
    no_stopwords_list.append(filtered_words)

df['no_stopwords']=no_stopwords_list

#small example
print(f"\nBefore (with stopwords): {len(df['tokens'].iloc[0])} words")
print(df['tokens'].iloc[0][:15])
print(f"\nAfter (no stopwords): {len(df['no_stopwords'].iloc[0])} words")
print(df['no_stopwords'].iloc[0][:15])

Number of stopwords: 198

Before (with stopwords): 129 words
['the', 'venal', 'gentlemen', 'mahmoud', 'a', 'simple', 'employee', 'of', 'the', 'ministry', 'of', 'health', 'loves', 'lawyer', 'amal']

After (no stopwords): 75 words
['venal', 'gentlemen', 'mahmoud', 'simple', 'employee', 'ministry', 'health', 'loves', 'lawyer', 'amal', 'trains', 'office', 'teacher', 'ali', 'hammad']


## Step 6: Stemming


In [ ]:
snowball=SnowballStemmer('english')
snowball_stemmed_list=[]

for words in df['no_stopwords']:
    snowball_words=[snowball.stem(word) for word in words]
    snowball_stemmed_list.append(snowball_words)
df['snowball_stem']=snowball_stemmed_list

## Step 7: Create Final Processed Text


In [ ]:
processed_text_list=[]
for stemmed_words in df['snowball_stem']:
    joined_text=' '.join(stemmed_words)
    processed_text_list.append(joined_text)

df['processed_text']=processed_text_list

#small example
print(f"Original: {df['combined_text'].iloc[0][:100]}...")
print(f"\nProcessed: {df['processed_text'].iloc[0][:100]}...")

Original: The Venal Gentlemen Mahmoud, a simple employee of the Ministry of Health, loves lawyer "Amal", who t...

Processed: venal gentlemen mahmoud simpl employe ministri health love lawyer amal train offic teacher ali hamma...


## Step 8: Prepare Output Dataset

In [ ]:
output_df=df[[
    'doc_id',
    'title',
    'type',
    'year',
    'rating',
    'abstract',
    'processed_text'
]].copy()

output_df.rename(columns={
    'doc_id': 'doc_id',
    'abstract': 'original_abstract'
}, inplace=True)

#small example
print(f"\nOutput columns: {list(output_df.columns)}")
print(f"\nFirst 3 rows:")
output_df.head(3)


Output columns: ['doc_id', 'title', 'type', 'year', 'rating', 'original_abstract', 'processed_text']

First 3 rows:


,doc_id,title,type,year,rating,original_abstract,processed_text
0,1,The Venal Gentlemen,movie,1983.0,6.000,"Mahmoud, a simple employee of the Ministry of ...",venal gentlemen mahmoud simpl employe ministri...
1,2,The Passage,movie,2019.0,7.042,"A platoon of Commandos’ soldiers, lead by a fe...",passag platoon commando soldier lead fearless ...
2,3,Karmouz War,movie,2018.0,6.000,"Alexandria, Egypt, 1940. Three young Egyptians...",karmouz war alexandria egypt 1940 three young ...


## Step 9: Save Preprocessed Data

In [ ]:
import os
output_dir=os.path.dirname(OUTPUT_FILE)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

# Save to CSV
output_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"Total records: {len(output_df)}")

Total records: 5036


In [ ]:
#from google.colab import files
#files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>